In [0]:
%sql
SELECT
hospital_id,
department_id,
visit_date,
visit_type,
occupied_beds
FROM medical_insurance.gold.operational_metrics 

ORDER BY hospital_id, department_id, visit_date

In [0]:
%sql
CREATE OR REPLACE TABLE medical_insurance.gold.operational_metrics AS
WITH visit_operations AS (
  SELECT 
    v.hospital_id,
    v.department_id,
    v.visit_date,
    DATE_FORMAT(v.visit_date, 'EEEE') AS day_of_week,
    YEAR(v.visit_date) AS visit_year,
    MONTH(v.visit_date) AS visit_month,
    v.visit_type,
    COUNT(DISTINCT v.visit_id) AS visit_count,
    COUNT(DISTINCT v.patient_id) AS unique_patients,
    COUNT(DISTINCT v.doctor_id) AS doctors_on_duty,
    AVG(v.waiting_time) AS avg_waiting_time_minutes,
    MIN(v.waiting_time) AS min_waiting_time,
    MAX(v.waiting_time) AS max_waiting_time,
    COUNT(DISTINCT CASE WHEN v.visit_status = 'Completed' THEN v.visit_id END) AS completed_visits,
    COUNT(DISTINCT CASE WHEN v.visit_status = 'Cancelled' THEN v.visit_id END) AS cancelled_visits
  FROM medical_insurance.silver.visit_silver v
  GROUP BY v.hospital_id, v.department_id, v.visit_date, v.visit_type
),
bed_operations AS (
  SELECT 
    b.hospital_id,
    b.department_id,
    COUNT(DISTINCT b.bed_id) AS total_beds,
    COUNT(DISTINCT CASE WHEN b.availability_status = 'Available' THEN b.bed_id END) AS available_beds,
    COUNT(DISTINCT CASE WHEN b.availability_status = 'Occupied' THEN b.bed_id END) AS occupied_beds,
    COUNT(DISTINCT CASE WHEN b.bed_type = 'ICU' THEN b.bed_id END) AS icu_beds,
    COUNT(DISTINCT CASE WHEN b.bed_type = 'Standard' THEN b.bed_id END) AS standard_beds
  FROM medical_insurance.silver.bed_silver b
  GROUP BY b.hospital_id, b.department_id
),
icu_operations AS (
  SELECT 
    hospital_id,
    occupied_beds AS icu_occupied,
    available_beds AS icu_available,
    update_time,
    ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY update_time DESC) AS rn
  FROM medical_insurance.silver.icu_status_silver
),
icu_latest AS (
  SELECT 
    hospital_id,
    icu_occupied AS latest_icu_occupied,
    icu_available AS latest_icu_available,
    update_time AS latest_update_time
  FROM icu_operations
  WHERE rn = 1
),
referral_operations AS (
  SELECT 
    from_hospital_id AS hospital_id,
    referral_date,
    COUNT(DISTINCT referral_id) AS referrals_sent,
    COUNT(DISTINCT CASE WHEN referral_status = 'Completed' THEN referral_id END) AS completed_referrals_sent,
    COUNT(DISTINCT CASE WHEN referral_status = 'Pending' THEN referral_id END) AS pending_referrals_sent,
    0 AS referrals_received,
    0 AS completed_referrals_received,
    0 AS pending_referrals_received
  FROM medical_insurance.silver.referral_silver
  GROUP BY from_hospital_id, referral_date
  UNION ALL
  SELECT 
    to_hospital_id AS hospital_id,
    referral_date,
    0 AS referrals_sent,
    0 AS completed_referrals_sent,
    0 AS pending_referrals_sent,
    COUNT(DISTINCT referral_id) AS referrals_received,
    COUNT(DISTINCT CASE WHEN referral_status = 'Completed' THEN referral_id END) AS completed_referrals_received,
    COUNT(DISTINCT CASE WHEN referral_status = 'Pending' THEN referral_id END) AS pending_referrals_received
  FROM medical_insurance.silver.referral_silver
  GROUP BY to_hospital_id, referral_date
),
schedule_operations AS (
  SELECT 
    d.hospital_id,
    s.shift_date,
    COUNT(DISTINCT s.doctor_id) AS scheduled_doctors,
    COUNT(DISTINCT s.schedule_id) AS total_shifts
  FROM medical_insurance.silver.doctor_schedule_silver s
  JOIN medical_insurance.silver.doctor_silver d ON s.doctor_id = d.doctor_id
  GROUP BY d.hospital_id, s.shift_date
),
hospital_info AS (
  SELECT 
    h.hospital_id,
    h.hospital_name,
    h.hospital_type,
    h.governorate,
    h.total_beds AS hospital_capacity_beds,
    h.icu_capacity AS hospital_icu_capacity
  FROM medical_insurance.silver.hospital_silver h
),
department_info AS (
  SELECT 
    dept.department_id,
    dept.hospital_id,
    dept.department_name
  FROM medical_insurance.silver.department_silver dept
)
SELECT 
  vo.hospital_id,
  h.hospital_name,
  h.hospital_type,
  h.governorate,
  vo.department_id,
  d.department_name,
  vo.visit_date AS operation_date,
  vo.day_of_week,
  vo.visit_year,
  vo.visit_month,
  vo.visit_type,
  
  -- Visit metrics
  COALESCE(vo.visit_count, 0) AS total_visits,
  COALESCE(vo.unique_patients, 0) AS unique_patients,
  COALESCE(vo.doctors_on_duty, 0) AS doctors_on_duty,
  COALESCE(vo.completed_visits, 0) AS completed_visits,
  COALESCE(vo.cancelled_visits, 0) AS cancelled_visits,
  CASE 
    WHEN vo.visit_count > 0 THEN ROUND((vo.completed_visits * 100.0 / vo.visit_count), 2)
    ELSE 0
  END AS completion_rate_pct,
  
  -- Wait time metrics
  ROUND(COALESCE(vo.avg_waiting_time_minutes, 0), 2) AS avg_waiting_time_minutes,
  COALESCE(vo.min_waiting_time, 0) AS min_waiting_time_minutes,
  COALESCE(vo.max_waiting_time, 0) AS max_waiting_time_minutes,
  
  -- Bed metrics
  h.hospital_capacity_beds,
  COALESCE(bo.total_beds, 0) AS beds_tracked,
  COALESCE(bo.available_beds, 0) AS available_beds,
  COALESCE(bo.occupied_beds, 0) AS occupied_beds,
  CASE 
    WHEN h.hospital_capacity_beds > 0 THEN ROUND((bo.occupied_beds * 100.0 / h.hospital_capacity_beds), 2)
    ELSE 0
  END AS bed_occupancy_rate_pct,
  COALESCE(bo.icu_beds, 0) AS icu_beds_tracked,
  COALESCE(bo.standard_beds, 0) AS standard_beds_tracked,
  
  -- ICU metrics
  h.hospital_icu_capacity,
  COALESCE(icu.latest_icu_occupied, 0) AS current_icu_occupied,
  COALESCE(icu.latest_icu_available, 0) AS current_icu_available,
  CASE 
    WHEN h.hospital_icu_capacity > 0 THEN ROUND((icu.latest_icu_occupied * 100.0 / h.hospital_icu_capacity), 2)
    ELSE 0
  END AS icu_occupancy_rate_pct,
  icu.latest_update_time AS icu_last_updated,
  
  -- Referral metrics
  SUM(ref.referrals_sent) AS referrals_sent,
  SUM(ref.referrals_received) AS referrals_received,
  SUM(ref.completed_referrals_sent) AS completed_referrals_sent,
  SUM(ref.pending_referrals_sent) AS pending_referrals_sent,
  
  -- Staffing metrics
  COALESCE(so.scheduled_doctors, 0) AS doctors_scheduled,
  COALESCE(so.total_shifts, 0) AS total_shifts_scheduled,
  CASE 
    WHEN vo.doctors_on_duty > 0 AND vo.visit_count > 0 THEN ROUND((vo.visit_count * 1.0 / vo.doctors_on_duty), 2)
    ELSE 0
  END AS visits_per_doctor,
  
  -- Efficiency indicators
  CASE 
    WHEN vo.avg_waiting_time_minutes < 30 THEN 'Efficient'
    WHEN vo.avg_waiting_time_minutes BETWEEN 30 AND 60 THEN 'Moderate'
    WHEN vo.avg_waiting_time_minutes > 60 THEN 'Needs Improvement'
    ELSE 'Unknown'
  END AS wait_time_efficiency,
  
  CASE 
    WHEN bo.occupied_beds * 100.0 / NULLIF(h.hospital_capacity_beds, 0) > 85 THEN 'Critical'
    WHEN bo.occupied_beds * 100.0 / NULLIF(h.hospital_capacity_beds, 0) BETWEEN 70 AND 85 THEN 'High'
    WHEN bo.occupied_beds * 100.0 / NULLIF(h.hospital_capacity_beds, 0) < 70 THEN 'Normal'
    ELSE 'Unknown'
  END AS capacity_status,
  
  CASE 
    WHEN vo.visit_count > 100 THEN 'High Volume'
    WHEN vo.visit_count BETWEEN 50 AND 100 THEN 'Medium Volume'
    WHEN vo.visit_count < 50 THEN 'Low Volume'
    ELSE 'No Activity'
  END AS daily_volume_category,
  
  CURRENT_TIMESTAMP() AS created_at
  
FROM visit_operations vo
LEFT JOIN hospital_info h ON vo.hospital_id = h.hospital_id
LEFT JOIN department_info d ON vo.department_id = d.department_id AND vo.hospital_id = d.hospital_id
LEFT JOIN bed_operations bo ON vo.hospital_id = bo.hospital_id AND vo.department_id = bo.department_id
LEFT JOIN icu_latest icu ON vo.hospital_id = icu.hospital_id
LEFT JOIN referral_operations ref ON vo.hospital_id = ref.hospital_id AND vo.visit_date = ref.referral_date
LEFT JOIN schedule_operations so ON vo.hospital_id = so.hospital_id AND vo.visit_date = so.shift_date
GROUP BY 
  vo.hospital_id, h.hospital_name, h.hospital_type, h.governorate, vo.department_id, d.department_name,
  vo.visit_date, vo.day_of_week, vo.visit_year, vo.visit_month, vo.visit_type, vo.visit_count, vo.unique_patients,
  vo.doctors_on_duty, vo.completed_visits, vo.cancelled_visits, vo.avg_waiting_time_minutes, vo.min_waiting_time,
  vo.max_waiting_time, h.hospital_capacity_beds, bo.total_beds, bo.available_beds, bo.occupied_beds, bo.icu_beds,
  bo.standard_beds, h.hospital_icu_capacity, icu.latest_icu_occupied, icu.latest_icu_available, icu.latest_update_time,
  so.scheduled_doctors, so.total_shifts

In [0]:
%sql
-- Display sample records from operational metrics gold table
SELECT 
  hospital_name,
  operation_date,
  day_of_week,
  department_name,
  total_visits,
  avg_waiting_time_minutes,
  bed_occupancy_rate_pct,
  wait_time_efficiency,
  capacity_status
FROM medical_insurance.gold.operational_metrics
ORDER BY operation_date DESC, total_visits DESC
LIMIT 10

In [0]:
%sql
-- Summary statistics by hospital and month
SELECT 
  hospital_name,
  hospital_type,
  visit_year,
  visit_month,
  SUM(total_visits) AS total_visits,
  ROUND(AVG(avg_waiting_time_minutes), 2) AS avg_wait_time,
  ROUND(AVG(bed_occupancy_rate_pct), 2) AS avg_bed_occupancy,
  ROUND(AVG(icu_occupancy_rate_pct), 2) AS avg_icu_occupancy,
  ROUND(AVG(completion_rate_pct), 2) AS avg_completion_rate,
  SUM(referrals_sent) AS total_referrals_sent,
  SUM(referrals_received) AS total_referrals_received
FROM medical_insurance.gold.operational_metrics
GROUP BY hospital_name, hospital_type, visit_year, visit_month
ORDER BY visit_year DESC, visit_month DESC, total_visits DESC

In [0]:
%sql
select * from medical_insurance.gold.operational_metrics 
where  hospital_id = "H005" and operation_date = "2024-06-05"